(ode:integration-schemes:mechanical-system)=
# Time-integration of a mechanical system



Reference to:

* [Modal and numerical methods for mechanical systems - continuum mechanics](https://basics2022.github.io/bbooks-physics-continuum-mechanics/ch/solids/small-displacements-numerics.html)
* [Natural modes in structures - blog](https://basics2022.github.io/myblog/posts/modes-in-structures/)

## Example equation: torsion of elastic beam

Here, the PDE governing the torsional dynamics of a linear homogeneous elastic beam is used as a toy problem

$$I \partial_{tt} \theta - \partial_x \left( GJ \partial_{x} \theta \right) = m $$

supplied with proper boundary and initial conditions. Here the clamped-free beam is studied, with prescribed torsional moment $M(t)$ ath teh "free" end so that the boundary conditions read

$$\begin{aligned}
 \theta(x=0, t) & = 0 \\
 GJ \partial_x \theta(x=\ell, t) & = M(t) \ .
\end{aligned}$$

See
* [Continuum mechanics: Beams](https://basics2022.github.io/bbooks-physics-continuum-mechanics/ch/solids/beams.html)
* [Continuum mechanics: Timoshenko beam](https://basics2022.github.io/bbooks-physics-continuum-mechanics/ch/solids/beams/timoshenko.html)
* [Finite element methods for beam structures](https://basics2022.github.io/bbooks-physics-continuum-mechanics/ch/solids/small-displacements-fem-beams.html)


### Exact solution

### Finite element model

Semi-discretized model in space

$$\begin{aligned}
 \int_{z=0}^{\ell} w(z) m(z,t) \, dz
 & = \int_{z=0}^{\ell} w(z) \left\{ I \partial_{tt} \theta - \partial_x \left(GJ \partial_x \theta \right) \right\} \, dz = \\
 & = \int_{z=0}^{\ell} w(z) I \partial_{tt} \theta(z,t) \, dz + \int_{z=0}^{\ell} \partial_x w(z) GJ \partial_x \theta(z,t) \, dz - \left.\left[ w(x) GJ \partial_x \theta(x,t) \right]\right|_{z=0}^{\ell} \ .
\end{aligned}$$

With the boundary conditions of the model, the FEM approximation of the problem becomes

$$\begin{aligned}
 \int_{z=0}^{\ell} w(z) m(z,t) \, dz + w(\ell) M(t) 
 & = \int_{z=0}^{\ell} w(z) I \partial_{tt} \theta(z,t) \, dz + \int_{z=0}^{\ell} \partial_x w(z) GJ \partial_x \theta(z,t) \, dz \ .
\end{aligned}$$

In [10]:
import numpy as np
from scipy.sparse import csr_matrix

def assemble_fem_matrices(L, N, I_coeff, GJ_coeff):
    """
    Assembles Sparse Mass and Stiffness matrices for P1 Lagrange elements
    based on 'Screenshot from 2026-05-14 18-59-39.png'.
    """
    # Grid setup
    nodes = np.linspace(0, L, N + 1)
    h = L / N  # element width
    num_nodes = N + 1

    # Local Element Matrices for P1 Lagrange
    # Mass Matrix Local: (h/6) * [[2, 1], [1, 2]]
    m_local = (I_coeff * h / 6.0) * np.array([[2.0, 1.0], 
                                              [1.0, 2.0]])
    
    # Stiffness Matrix Local: (1/h) * [[1, -1], [-1, 1]]
    k_local = (GJ_coeff / h) * np.array([[1.0, -1.0], 
                                         [-1.0, 1.0]])

    # Global Assembly using COO format for efficiency
    rows = []
    cols = []
    mass_data = []
    stiff_data = []

    for i in range(N):
        # Indices for the current element (nodes i and i+1)
        idx = [i, i + 1]
        
        for r in range(2):
            for c in range(2):
                rows.append(idx[r])
                cols.append(idx[c])
                mass_data.append(m_local[r, c])
                stiff_data.append(k_local[r, c])

    # Convert to CSR (Compressed Sparse Row) format
    M_sparse = csr_matrix((mass_data, (rows, cols)), shape=(num_nodes, num_nodes))
    K_sparse = csr_matrix((stiff_data, (rows, cols)), shape=(num_nodes, num_nodes))

    return M_sparse, K_sparse

def get_partitioned_matrices(L, N, I_coeff, GJ_coeff, dirichlet_indices):
    # 1. Reuse the assembly logic to get global matrices
    # (Assuming the assembly function from the previous step is defined)
    M_global, K_global = assemble_fem_matrices(L, N, I_coeff, GJ_coeff)
    
    num_nodes = N + 1
    
    # 2. Define Node Sets
    # Node 0 is Dirichlet (index 0), all others are Free
    all_indices = np.arange(num_nodes)
    free_indices = np.setdiff1d(all_indices, dir_indices)
    
    # 3. Partitioning: Extract the FF block (Free-Free)
    # This effectively removes the first row and first column
    M_ff = M_global[free_indices, :][:, free_indices]
    K_ff = K_global[free_indices, :][:, free_indices]
    
    return M_ff, K_ff, free_indices


# Example parameters
L = 1.0           # Length of the domain
N = 10            # Number of elements
I_val = 1.0       # Inertia coefficient (from image)
GJ_val = 1.0      # Torsional stiffness (from image)
dir_indices = [0] # Dirichlet indices of the nodes with prescribed displacement
Mff, Kff, free_indices = get_partitioned_matrices(L, N, I_val, GJ_val, dir_indices)

print(f"Mass Matrix Shape: {Mff.shape}")
print(f"Mass      Matrix (First 3x3 block):\n{Mff.toarray()[:3, :3]}")
print(f"Stiffness Matrix (First 3x3 block):\n{Kff.toarray()[:3, :3]}")
print(f"Sum(Mass Matrix): {np.sum(Mff.toarray())}")


Mass Matrix Shape: (10, 10)
Mass      Matrix (First 3x3 block):
[[0.06666667 0.01666667 0.        ]
 [0.01666667 0.06666667 0.01666667]
 [0.         0.01666667 0.06666667]]
Stiffness Matrix (First 3x3 block):
[[ 20. -10.   0.]
 [-10.  20. -10.]
 [  0. -10.  20.]]
Sum(Mass Matrix): 0.9333333333333333


#### Spectral decomposition of the undamped system

In [16]:
import scipy.linalg as la

def solve_spectral_decomposition(M_ff, K_ff):
    """
    Solves the generalized eigenvalue problem K*v = lambda*M*v
    using the partitioned matrices derived from the model.
    """
    # Convert sparse matrices to dense for the standard solver
    # (For very large N, use scipy.sparse.linalg.eigsh instead)
    M_dense = M_ff.toarray()
    K_dense = K_ff.toarray()

    # Solve generalized eigenvalue problem
    # eigh returns eigenvalues in ascending order
    eigenvalues, eigenvectors = la.eigh(K_dense, M_dense)
    
    # Natural frequencies (rad/s) are the square roots of eigenvalues
    # omegas = np.sqrt(np.maximum(eigenvalues, 0))
    print(eigenvalues)
    s = ( -np.real(eigenvalues) - 1j* np.imag(eigenvalues))**.5
    
    return eigenvalues, eigenvectors, s # omegas

# --- Execution ---
# Using the M_red and K_red from the previous partitioning step
evals, evecs, frequencies = solve_spectral_decomposition(Mff, Kff)

print(f"First 3 Natural Frequencies (rad/s): {frequencies[:3]}")

[   2.47247865   22.62052505   64.91651253  133.49917214  234.71119998
  376.36887101  564.28780011  792.22634461 1023.09340791 1178.10853335]
First 3 Natural Frequencies (rad/s): [0.+1.57241173j 0.+4.75610398j 0.+8.05707841j]


#### Modal structural damping

See [Continuum mechanics: structural damping](https://basics2022.github.io/bbooks-physics-continuum-mechanics/ch/solids/structural-damping.html)

In [1]:

import numpy as np
